# Hossain Group Employee Turnover Analytics

Synthetic Bangladesh-based HR data for turnover calculation, department benchmarking, and exit-reason analysis.


In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

candidates = list(Path('/kaggle/input').rglob('employee_master.csv'))
if candidates:
    DATA_PATH = candidates[0]
else:
    DATA_PATH = Path('../data/raw/employee_master.csv')
print('Using:', DATA_PATH)
df = pd.read_csv(DATA_PATH, parse_dates=['Join_Date', 'Exit_Date'])
df.head()


In [ ]:
analysis_start = pd.Timestamp('2025-01-01')
analysis_end = pd.Timestamp('2026-06-30')
months = pd.date_range(analysis_start, analysis_end, freq='MS')

def active_on(d):
    return (df['Join_Date'] <= d) & (df['Exit_Date'].isna() | (df['Exit_Date'] >= d))

rows = []
for month_start in months:
    month_end = month_start + pd.offsets.MonthEnd(0)
    opening = active_on(month_start).sum()
    closing = ((df['Join_Date'] <= month_end) & (df['Exit_Date'].isna() | (df['Exit_Date'] > month_end))).sum()
    exits = df['Exit_Date'].between(month_start, month_end).sum()
    hires = df['Join_Date'].between(month_start, month_end).sum()
    avg_hc = (opening + closing) / 2
    rows.append([month_start, opening, hires, exits, closing, avg_hc, exits / avg_hc if avg_hc else 0])

monthly = pd.DataFrame(rows, columns=['MonthStart','OpeningHC','Hires','Exits','ClosingHC','AverageHC','TurnoverRate'])
monthly


In [ ]:
period_exits = df['Exit_Date'].between(analysis_start, analysis_end).sum()
average_hc = monthly['AverageHC'].mean()
period_turnover = period_exits / average_hc
annualized_turnover = period_turnover * 12 / len(monthly)
pd.Series({
    'Total exits': period_exits,
    'Average headcount': round(average_hc, 2),
    'Period turnover': period_turnover,
    'Annualized turnover': annualized_turnover,
})


In [ ]:
ax = monthly.plot(x='MonthStart', y='TurnoverRate', marker='o', figsize=(12,5), legend=False)
ax.set_title('Monthly Employee Turnover Rate')
ax.set_ylabel('Turnover Rate')
ax.yaxis.set_major_formatter(lambda x, pos: f'{x:.1%}')
plt.show()


In [ ]:
exited = df[df['Exit_Date'].between(analysis_start, analysis_end)].copy()
dept = exited.groupby('Department').size().sort_values(ascending=False)
fig, ax = plt.subplots(figsize=(12,5))
ax.bar(dept.index.astype(str), dept.to_numpy())
ax.set_title('Exits by Department')
ax.set_ylabel('Employees Exited')
ax.tick_params(axis='x', rotation=45)
fig.tight_layout()
plt.show()


## HR Interpretation

- Compare turnover with department headcount before ranking risk.
- Review exit reasons, manager patterns, compensation, career mobility, and contract completion.
- Use turnover as a diagnostic signal, not a standalone judgement of HR performance.
